# 10_feature_eda_260513

Descriptive conservative feature EDA with CSV audit tables and matplotlib figures.

In [1]:
from pathlib import Path
import subprocess, os, zipfile, json, math, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

def set_korean_font():
    preferred_fonts = ["Malgun Gothic", "Noto Sans CJK KR", "Noto Sans KR", "NanumGothic", "AppleGothic"]
    available = {f.name for f in fm.fontManager.ttflist}
    for font in preferred_fonts:
        if font in available:
            plt.rcParams["font.family"] = font
            plt.rcParams["axes.unicode_minus"] = False
            return font
    plt.rcParams["axes.unicode_minus"] = False
    return None
KOREAN_FONT_USED = set_korean_font()
DEFAULT_FIGSIZE = (12, 7)
DEFAULT_DPI = 170

def annotate_bars_count_pct(ax, bars, counts, denominators, fontsize=10):
    for bar, count, denom in zip(bars, counts, denominators):
        pct = (count / denom * 100) if denom else 0
        label = f"{int(count):,} ({pct:.1f}%)"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label, ha="center", va="bottom", fontsize=fontsize)

def st(v): return "PASS" if bool(v) else "FAIL"
def inside(p, parent): return str(Path(p).resolve()).lower().startswith(str(Path(parent).resolve()).lower())
def choose_folder(base):
    base=Path(base)
    target = base / f"run_{RUN_TS}" if base.exists() and any(base.iterdir()) else base
    target.mkdir(parents=True, exist_ok=True)
    return target

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPECTED_ROOTS = {"C:/Code/ott-churn-prediction", "C:\\Code\\ott-churn-prediction"}
actual_repo_root_raw = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
repo_root_match = actual_repo_root_raw in EXPECTED_ROOTS
print("repo root:", actual_repo_root_raw)
if not repo_root_match:
    raise SystemExit(f"STOP: repo root mismatch: {actual_repo_root_raw}")
REPO = Path(actual_repo_root_raw).resolve(); PARK = REPO / "park.ingyeom"
NOTE_PATH = PARK / "note.md"
NB_PATH = PARK / "notebook" / "10_feature_eda_260513" / "10_feature_eda_260513.ipynb"
OUT = choose_folder(PARK / "reports" / "eda" / "10_feature_eda_260513")
FIG = choose_folder(PARK / "reports" / "figures" / "10_feature_eda_260513")
ZIP_DIR = PARK / "zip"; ZIP_DIR.mkdir(exist_ok=True)
ZIP_PATH = ZIP_DIR / "10_feature_eda_260513_review_package.zip"
print("actual_output_folder:", OUT)
print("actual_figure_folder:", FIG)
print("KOREAN_FONT_USED:", KOREAN_FONT_USED)

P06 = PARK / "reports" / "audits" / "06_common_preprocessing_and_final_cohort_260513"
P05 = PARK / "reports" / "audits" / "05b_column_role_dictionary_patch_260513"
P07 = PARK / "reports" / "audits" / "07_AARRR_feature_mapping_260513"
P08 = PARK / "reports" / "eda" / "08_promotion_vs_nonpromotion_eda_260513" / "run_20260514_022322"
P08B = PARK / "reports" / "eda" / "08b_promotion_vs_nonpromotion_eda_audit_patch_260513"
P09 = PARK / "reports" / "eda" / "09_promotion_repurchase_2x2_eda_260513"
P09B_PREF = PARK / "reports" / "audits" / "09b_raw_view_window_validation_260514" / "run_20260514_130402"
P09B_FALL = PARK / "reports" / "audits" / "09b_raw_view_window_validation_260514"
req09b_names = ["09b_final_checks.csv","09b_core_usage_recalculation_comparison.csv","09b_day21_plus_leakage_contrast_test.csv","09b_window_validation_decision.csv","09b_open_risks_for_next_steps.csv"]
def all_pass(path):
    if not path.exists(): return False
    d=pd.read_csv(path); return "status" in d and (d["status"].astype(str).str.upper()=="PASS").all()
P09B = P09B_PREF if P09B_PREF.exists() and all((P09B_PREF/n).exists() for n in req09b_names) and all_pass(P09B_PREF/"09b_final_checks.csv") else P09B_FALL
required = {
"source":[PARK/"data"/"(광일)Membership_v2_with_derived_features.csv"],
"06":[P06/"06_primary_main_cohort_index.csv",P06/"06_primary_main_cohort_conservative_features.csv",P06/"06_feature_policy_from_05b.csv",P06/"06_final_checks.csv"],
"05b":[P05/"05b_canonical_column_role_dictionary.csv",P05/"05b_conservative_safe_candidate_columns.csv",P05/"05b_review_required_columns.csv",P05/"05b_forbidden_drop_columns.csv"],
"07":[P07/"07_AARRR_mapping_conservative_features.csv",P07/"07_AARRR_feature_mapping_all_columns.csv",P07/"07_final_checks.csv"],
"08":[P08/"08_final_checks.csv"],
"08b":[P08B/"08b_final_checks.csv",P08B/"08b_interpretation_guardrail.csv",P08B/"08b_decision_summary.csv"],
"09":[P09/"09_final_checks.csv",P09/"09_2x2_cohort_definition.csv",P09/"09_within_promotion_target_difference_summary.csv",P09/"09_within_nonpromotion_target_difference_summary.csv",P09/"09_cross_group_target_signal_comparison.csv",P09/"09_top_target_signals_by_group.csv",P09/"09_08_vs_09_contrast_summary.csv",P09/"09_open_risks_for_next_steps.csv"],
"09b":[P09B/n for n in req09b_names]}
pre=[]
def add_pre(name, value, status=None, notes=""):
    pre.append({"check_name":name,"status":status or st(value),"value":value,"notes":notes,"actual_output_folder":str(OUT),"actual_figure_folder":str(FIG)})
missing=[]
for g,ps in required.items():
    for p in ps:
        if not p.exists(): missing.append(str(p))
add_pre("expected_repo_root","C:/Code/ott-churn-prediction","PASS")
add_pre("actual_repo_root",actual_repo_root_raw,"PASS")
add_pre("repo_root_match",repo_root_match)
add_pre("source_file_exists",required["source"][0].exists())
for g in ["06","05b","07","08","08b","09","09b"]: add_pre(f"required_{g}_files_exist", all(p.exists() for p in required[g]))
add_pre("detected_09b_output_folder",str(P09B),"PASS" if all(p.exists() for p in required["09b"]) else "FAIL")
add_pre("note_md_exists",NOTE_PATH.exists())
add_pre("source_file_inside_park_ingyeom",inside(required["source"][0],PARK))
add_pre("output_folder_inside_park_ingyeom",inside(OUT,PARK))
add_pre("figure_folder_inside_park_ingyeom",inside(FIG,PARK))
add_pre("notebook_inside_park_ingyeom",inside(NB_PATH,PARK))
add_pre("zip_folder_inside_park_ingyeom",inside(ZIP_DIR,PARK))
can_proceed=not missing and all(r["status"]=="PASS" for r in pre if r["check_name"]!="note_md_exists")
add_pre("can_proceed",can_proceed,"PASS" if can_proceed else "FAIL","; ".join(missing[:5]))
pd.DataFrame(pre).to_csv(OUT/"10_preflight_input_validation.csv",index=False,encoding="utf-8-sig")
if not can_proceed:
    (OUT/"README.md").write_text("# 10_feature_eda_260513\n\nPreflight failed.\n"+"\n".join(missing),encoding="utf-8")
    raise SystemExit("STOP: preflight failed")
print("detected 09b output folder:", P09B)


repo root: C:/Code/ott-churn-prediction
actual_output_folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\eda\10_feature_eda_260513
actual_figure_folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\figures\10_feature_eda_260513
KOREAN_FONT_USED: Malgun Gothic
detected 09b output folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\09b_raw_view_window_validation_260514\run_20260514_130402


In [2]:
# Load data and create base distribution/catalog outputs.
source = pd.read_csv(required["source"][0])
cohort_idx = pd.read_csv(required["06"][0])
cf = pd.read_csv(required["06"][1])
cons05 = pd.read_csv(required["05b"][1])
review05 = pd.read_csv(required["05b"][2])
forbid05 = pd.read_csv(required["05b"][3])
map07 = pd.read_csv(required["07"][0])
final09b = pd.read_csv(P09B/"09b_final_checks.csv")
within_promo09 = pd.read_csv(P09/"09_within_promotion_target_difference_summary.csv")
within_nonpromo09 = pd.read_csv(P09/"09_within_nonpromotion_target_difference_summary.csv")
cross09 = pd.read_csv(P09/"09_cross_group_target_signal_comparison.csv")
top09 = pd.read_csv(P09/"09_top_target_signals_by_group.csv")
contrast09 = pd.read_csv(P09/"09_08_vs_09_contrast_summary.csv")
contrast09 = contrast09[contrast09["feature_name"].astype(str)!="SUMMARY"].copy()
review_cols=set(review05.get("column_name",pd.Series(dtype=str)).dropna().astype(str))
forbid_cols=set(forbid05.get("column_name",pd.Series(dtype=str)).dropna().astype(str))
features=[c for c in cons05["column_name"].dropna().astype(str) if c in cf.columns and c not in review_cols and c not in forbid_cols]
order=[c for c in map07["column_name"].astype(str) if c in features and not c.startswith("__summary")]
features=order+[c for c in features if c not in order]
id_cols=["source_row_number","USER_KEY","is_promotion","is_repurchase"]
df=cf[id_cols+features].copy()
for c in ["is_promotion","is_repurchase"]: df[c]=pd.to_numeric(df[c],errors="coerce").astype("Int64")
cohort_order=[(0,1,"비프로모션 재구매","nonpromotion_repurchase"),(0,0,"비프로모션 미재구매","nonpromotion_nonrepurchase"),(1,1,"프로모션 재구매","promotion_repurchase"),(1,0,"프로모션 미재구매","promotion_nonrepurchase")]
krmap={(p,r):kr for p,r,kr,en in cohort_order}; enmap={(p,r):en for p,r,kr,en in cohort_order}
cohort_colors={"비프로모션 재구매":"#1D9E75","비프로모션 미재구매":"#D4537E","프로모션 재구매":"#378ADD","프로모션 미재구매":"#F0A33A"}
df["cohort_name_kr"]=[krmap[(int(p),int(r))] for p,r in zip(df.is_promotion,df.is_repurchase)]
df["cohort_name"]=[enmap[(int(p),int(r))] for p,r in zip(df.is_promotion,df.is_repurchase)]
print("primary main cohort row count:", len(df))
print("conservative feature count:", len(features))

def num(s): return pd.to_numeric(s,errors="coerce")
def isbin(s):
    vals=set(num(s).dropna().unique().tolist()); return vals.issubset({0,1}) and len(vals)<=2
def quant(s,p):
    x=num(s).dropna(); return float(x.quantile(p)) if len(x) else np.nan
def topshare(s,pct):
    x=num(s).dropna()
    if len(x)==0 or (x<0).any(): return np.nan
    total=x.sum()
    if total==0: return 0.0
    return float(x.sort_values(ascending=False).head(max(1,math.ceil(len(x)*pct))).sum()/total)
def ftype(f):
    if f.startswith("is_"): return "binary"
    if "watch_session" in f: return "count"
    if "watch_time" in f: return "continuous_time"
    if "ratio" in f or "over_50pct" in f: return "ratio"
    if "diff_between" in f: return "retention_change"
    if "gap" in f: return "gap"
    return "unknown"
status09b_core=bool(((final09b.check_name=="core_weekly_usage_matches_master_day0_20")&(final09b.status.astype(str).str.upper()=="PASS")).any())
checks=[]
def add_check(n,ok,val="",note=""): checks.append({"check_name":n,"status":st(ok),"value":val,"notes":note,"actual_output_folder":str(OUT),"actual_figure_folder":str(FIG)})
add_check("primary main cohort row count = 23,079",len(df)==23079,len(df)); add_check("conservative feature table row count = 23,079",len(cf)==23079,len(cf)); add_check("conservative feature count = 22",len(features)==22,len(features))
for c in ["is_promotion","is_repurchase","USER_KEY"]: add_check(f"{c} exists",c in cf.columns)
add_check("no review columns used as conservative feature columns",not(set(features)&review_cols),", ".join(sorted(set(features)&review_cols))); add_check("no forbidden columns used as conservative feature columns",not(set(features)&forbid_cols),", ".join(sorted(set(features)&forbid_cols)))
add_check("no repurchase_score column","repurchase_score" not in cf.columns); add_check("no churn_risk column","churn_risk" not in cf.columns); add_check("no prediction columns",not any("pred" in c.lower() for c in cf.columns)); add_check("09b core usage validation passed",status09b_core); add_check("09b day21+ contrast exists",(P09B/"09b_day21_plus_leakage_contrast_test.csv").exists()); add_check("09 top signal files exist",all((P09/n).exists() for n in ["09_within_promotion_target_difference_summary.csv","09_within_nonpromotion_target_difference_summary.csv","09_top_target_signals_by_group.csv"]))
pd.DataFrame(checks).to_csv(OUT/"10_cohort_feature_consistency_check.csv",index=False,encoding="utf-8-sig")
map07c=map07[~map07.column_name.astype(str).str.startswith("__summary")].copy(); meta=map07c.set_index("column_name").to_dict("index")
top_features=set(top09.feature_name.dropna().astype(str)); common=set(top09.loc[top09.group_context.astype(str).str.contains("common|stronger",case=False,na=False),"feature_name"].dropna().astype(str))
preferred=["watch_time(min)_w3","watch_session_w3","is_only_w1","is_w1_over_50pct","retention_w3_ratio","retention_w2_ratio","diff_between_w3_w1","diff_between_w3_w2","diff_between_w2_w1","watch_time(min)_w2","watch_session_w2","is_cold_start_3d","is_cold_start_7d"]
focus=[f for f in preferred if f in features]
cat=[]
for f in features:
    m=meta.get(f,{})
    cat.append({"feature_name":f,"dtype":str(cf[f].dtype),"feature_type":ftype(f),"AARRR_stage_primary":m.get("AARRR_stage_primary","unknown"),"feature_family":m.get("feature_family","unknown"),"timing_family":m.get("timing_family","unknown"),"selected_for_deep_dive":"yes" if f in focus else "no","reason_for_deep_dive":"09 top/common signal or requested Step 10 focus feature" if f in focus else "catalog only","appeared_in_09_top_signals":"yes" if f in top_features else "no","appeared_in_09_common_or_different_signals":"yes" if f in common else "no","safe_interpretation_boundary":"descriptive row-level association only; no causality, no p-value, no final threshold"})
catalog=pd.DataFrame(cat); catalog.to_csv(OUT/"10_feature_eda_catalog.csv",index=False,encoding="utf-8-sig")
print("top focus features:", ", ".join(focus))
glob=[]
for f in features:
    s=num(df[f]); x=s.dropna(); zero=int((s==0).sum()); pos=int((s>0).sum()); q99=quant(s,.99); mx=float(x.max()) if len(x) else np.nan
    glob.append({"feature_name":f,"n":int(s.notna().sum()),"missing_count":int(s.isna().sum()),"missing_rate":float(s.isna().mean()),"mean":float(x.mean()) if len(x) else np.nan,"std":float(x.std()) if len(x) else np.nan,"min":float(x.min()) if len(x) else np.nan,"q01":quant(s,.01),"q05":quant(s,.05),"q10":quant(s,.10),"q25":quant(s,.25),"median":quant(s,.5),"q75":quant(s,.75),"q90":quant(s,.9),"q95":quant(s,.95),"q99":q99,"max":mx,"zero_count":zero,"zero_rate":zero/len(s),"positive_count":pos,"positive_rate":pos/len(s),"unique_count":int(x.nunique()),"binary_flag":isbin(s),"high_zero_inflation_flag":zero/len(s)>=.5,"extreme_outlier_flag":pd.notna(q99) and q99!=0 and pd.notna(mx) and mx/q99>=3,"top_1pct_share_of_total":topshare(s,.01),"note":"descriptive only"})
global_df=pd.DataFrame(glob); global_df.to_csv(OUT/"10_global_feature_distribution_profile.csv",index=False,encoding="utf-8-sig")
prof=[]
for p,r,kr,en in cohort_order:
    g=df[(df.is_promotion==p)&(df.is_repurchase==r)]
    for f in features:
        s=num(g[f]); x=s.dropna(); prof.append({"cohort_name":en,"cohort_name_kr":kr,"is_promotion":p,"is_repurchase":r,"feature_name":f,"n":int(s.notna().sum()),"mean":float(x.mean()) if len(x) else np.nan,"std":float(x.std()) if len(x) else np.nan,"min":float(x.min()) if len(x) else np.nan,"q05":quant(s,.05),"q10":quant(s,.1),"q25":quant(s,.25),"median":quant(s,.5),"q75":quant(s,.75),"q90":quant(s,.9),"q95":quant(s,.95),"max":float(x.max()) if len(x) else np.nan,"zero_rate":float((s==0).mean()),"positive_rate":float((s>0).mean()),"top_1pct_share_of_total":topshare(s,.01),"distribution_note":"2x2 descriptive distribution only"})
prof_df=pd.DataFrame(prof); prof_df.to_csv(OUT/"10_2x2_feature_distribution_profile.csv",index=False,encoding="utf-8-sig")


primary main cohort row count: 23079
conservative feature count: 22
top focus features: watch_time(min)_w3, watch_session_w3, is_only_w1, is_w1_over_50pct, retention_w3_ratio, retention_w2_ratio, diff_between_w3_w1, diff_between_w3_w2, diff_between_w2_w1, watch_time(min)_w2, watch_session_w2, is_cold_start_3d, is_cold_start_7d


In [3]:
# Deep dives, audits, bins, week, handoff tables.
promo=within_promo09.set_index("feature_name").to_dict("index"); nonpromo=within_nonpromo09.set_index("feature_name").to_dict("index"); cross=cross09.set_index("feature_name").to_dict("index")
deep=[]
for f in focus:
    gd=global_df.set_index("feature_name").loc[f]; m=meta.get(f,{})
    deep.append({"feature_name":f,"feature_type":ftype(f),"AARRR_stage_primary":m.get("AARRR_stage_primary","unknown"),"feature_family":m.get("feature_family","unknown"),"why_selected":"requested focus and/or Step 09 target-internal signal","09_within_promotion_SMD":promo.get(f,{}).get("simple_standardized_mean_difference",np.nan),"09_within_nonpromotion_SMD":nonpromo.get(f,{}).get("simple_standardized_mean_difference",np.nan),"09_signal_pattern_label":cross.get(f,{}).get("signal_pattern_label","not_available"),"overall_distribution_shape":"binary proportion" if ftype(f)=="binary" else ("zero-inflated" if gd.zero_rate>=.5 else "continuous/count skew check"),"zero_inflation_summary":f"overall zero rate {gd.zero_rate:.1%}","outlier_influence_summary":f"top 1pct share {gd.top_1pct_share_of_total:.1%}" if pd.notna(gd.top_1pct_share_of_total) else "not applicable for negative-valued feature","2x2_pattern_summary":cross.get(f,{}).get("safe_interpretation","see 2x2 distribution table"),"safe_claim":"row-level descriptive distribution difference observed; validate later in modeling if used","unsafe_claim":"causal effect, statistical significance, model performance, or final segment threshold","recommended_next_use":"modeling_candidate" if f not in ["is_only_w1","is_w1_over_50pct"] else "segmentation_candidate_later"})
deep_df=pd.DataFrame(deep); deep_df.to_csv(OUT/"10_focus_feature_deep_dive_summary.csv",index=False,encoding="utf-8-sig")
zero=[]
for f in features:
    rates=[]; prs=[]
    for p,r,kr,en in cohort_order:
        g=df[(df.is_promotion==p)&(df.is_repurchase==r)]; s=num(g[f]); rates.append((en,float((s==0).mean())))
        if isbin(s): prs.append((en,float((s==1).mean())))
    gap=max(v for _,v in rates)-min(v for _,v in rates); pgap=(max(v for _,v in prs)-min(v for _,v in prs)) if prs else np.nan
    zero.append({"feature_name":f,"feature_type":ftype(f),"overall_zero_rate":float((num(df[f])==0).mean()),"zero_rate_by_2x2":json.dumps({k:round(v,6) for k,v in rates},ensure_ascii=False),"max_zero_rate_gap_between_2x2":gap,"binary_positive_rate_by_2x2":json.dumps({k:round(v,6) for k,v in prs},ensure_ascii=False) if prs else "","max_positive_rate_gap_between_2x2":pgap,"whether_signal_is_mainly_zero_nonzero":"yes" if gap>=.15 or (pd.notna(pgap) and pgap>=.15) else ("review" if gap>=.05 else "no"),"interpretation":"descriptive zero/nonzero or binary proportion audit only"})
zero_df=pd.DataFrame(zero); zero_df.to_csv(OUT/"10_zero_inflation_binary_audit.csv",index=False,encoding="utf-8-sig")
print("zero-inflation summary:", zero_df.sort_values("max_zero_rate_gap_between_2x2",ascending=False).head(5)[["feature_name","overall_zero_rate","max_zero_rate_gap_between_2x2"]].to_string(index=False))
out=[]
for f in features:
    s=num(df[f])
    if isbin(s): continue
    q99=quant(s,.99); mx=float(s.max()) if s.notna().any() else np.nan; top1=topshare(s,.01); top5=topshare(s,.05); cell=[]
    for p,r,kr,en in cohort_order: cell.append((en,topshare(df[(df.is_promotion==p)&(df.is_repurchase==r)][f],.01)))
    best=sorted(cell,key=lambda x:-1 if pd.isna(x[1]) else x[1],reverse=True)[0][0]
    sens="yes" if (pd.notna(top1) and top1>=.25) or (pd.notna(q99) and q99!=0 and pd.notna(mx) and mx/q99>=3) else ("review" if pd.notna(top1) and top1>=.15 else "no")
    out.append({"feature_name":f,"overall_mean":float(s.mean()),"overall_median":float(s.median()),"mean_median_gap":float(s.mean()-s.median()),"q99":q99,"max":mx,"max_to_q99_ratio":mx/q99 if pd.notna(q99) and q99!=0 else np.nan,"top_1pct_share_of_total":top1,"top_5pct_share_of_total":top5,"promotion_target_cell_with_highest_top1pct_share":best,"whether_mean_may_be_outlier_sensitive":sens,"recommended_summary_metric":"both" if sens in ["yes","review"] else "mean","interpretation":"descriptive outlier influence audit; no winsorization or transformation applied"})
outlier_df=pd.DataFrame(out); outlier_df.to_csv(OUT/"10_outlier_influence_audit.csv",index=False,encoding="utf-8-sig")
print("outlier influence summary:", outlier_df.sort_values("top_1pct_share_of_total",ascending=False).head(5)[["feature_name","top_1pct_share_of_total","recommended_summary_metric"]].to_string(index=False))

def bin_feature(series,f):
    x=num(series); typ=ftype(f)
    if typ=="binary": return x.map({0:"0",1:"1"}).fillna("missing")
    if "watch_session" in f:
        bins=[-np.inf,0,1,3,7,np.inf]; labs=["0","1","2-3","4-7","8+"]
        if x.max()<=3: bins=[-np.inf,0,1,np.inf]; labs=["0","1","2+"]
        return pd.cut(x,bins=bins,labels=labs,right=True).astype(str)
    if "watch_time" in f:
        pos=x[x>0]; qs=sorted(set([float(pos.quantile(.25)),float(pos.quantile(.5)),float(pos.quantile(.75))])) if len(pos) else []
        def lab(v):
            if pd.isna(v): return "missing"
            if v==0: return "0"
            if len(qs)>=3:
                if v<=qs[0]: return f"(0,{qs[0]:.0f}]"
                if v<=qs[1]: return f"({qs[0]:.0f},{qs[1]:.0f}]"
                if v<=qs[2]: return f"({qs[1]:.0f},{qs[2]:.0f}]"
                return f">{qs[2]:.0f}"
            return "positive"
        return x.map(lab)
    if "ratio" in f: return pd.cut(x,bins=[-np.inf,0,.25,.5,.75,1,np.inf],labels=["0","(0,0.25]","(0.25,0.5]","(0.5,0.75]","(0.75,1.0]",">1"],right=True).astype(str)
    if "diff_between" in f: return x.map(lambda v:"missing" if pd.isna(v) else ("negative" if v<0 else ("zero" if v==0 else "positive")))
    if "gap" in f: return pd.cut(x,bins=[-np.inf,0,1,3,np.inf],labels=["0","1","2-3","4+"],right=True).astype(str)
    return pd.qcut(x.rank(method="first"),q=min(4,x.notna().sum()),duplicates="drop").astype(str)
bins=[]
for f in focus:
    temp=df[["cohort_name","cohort_name_kr","is_promotion","is_repurchase"]].copy(); temp["bin_label"]=bin_feature(df[f],f)
    for (cohort,kr,b),g in temp.groupby(["cohort_name","cohort_name_kr","bin_label"],dropna=False):
        denom=int((temp.cohort_name==cohort).sum()); cnt=len(g); same=temp[(temp.bin_label==b)&(temp.is_promotion==g.is_promotion.iloc[0])]
        bins.append({"feature_name":f,"bin_label":str(b),"cohort_name":cohort,"cohort_name_kr":kr,"count":cnt,"percent_within_cohort":cnt/denom if denom else np.nan,"repurchase_rate_within_bin":float(same.is_repurchase.mean()) if len(same) else np.nan,"note":"exploratory EDA bin only; not final segment threshold"})
bin_df=pd.DataFrame(bins); bin_df.to_csv(OUT/"10_focus_feature_bin_distribution.csv",index=False,encoding="utf-8-sig")
sep=[]
for f in focus:
    sep.append({"feature_name":f,"within_promotion_shape_label":"descriptive shape review","within_nonpromotion_shape_label":"descriptive shape review","common_or_different_shape":cross.get(f,{}).get("signal_pattern_label","not_available"),"evidence":f"promotion SMD={promo.get(f,{}).get('simple_standardized_mean_difference',np.nan)}, nonpromotion SMD={nonpromo.get(f,{}).get('simple_standardized_mean_difference',np.nan)}","safe_interpretation":"descriptive separation shape only","caution":"no causality, no p-value, no final threshold"})
pd.DataFrame(sep).to_csv(OUT/"10_2x2_separation_shape_audit.csv",index=False,encoding="utf-8-sig")
# week and remaining tables
week=[]
for p,r,kr,en in cohort_order:
    g=df[(df.is_promotion==p)&(df.is_repurchase==r)]; tw=[num(g[f"watch_time(min)_w{i}"]) for i in [1,2,3]]; means=[float(s.mean()) for s in tw]; meds=[float(s.median()) for s in tw]; zeros=[float((s==0).mean()) for s in tw]
    pat="late_activation" if means[2]>means[0] and means[2]>means[1] else ("decreasing" if means[0]>means[1]>means[2] else "mixed")
    week.append({"cohort_name":en,"cohort_name_kr":kr,"is_promotion":p,"is_repurchase":r,"mean_w1":means[0],"mean_w2":means[1],"mean_w3":means[2],"median_w1":meds[0],"median_w2":meds[1],"median_w3":meds[2],"zero_rate_w1":zeros[0],"zero_rate_w2":zeros[1],"zero_rate_w3":zeros[2],"w2_minus_w1_mean":means[1]-means[0],"w3_minus_w1_mean":means[2]-means[0],"w3_minus_w2_mean":means[2]-means[1],"dominant_pattern":pat,"interpretation":"descriptive week progression only; no causality"})
week_df=pd.DataFrame(week); week_df.to_csv(OUT/"10_week_progression_pattern_audit.csv",index=False,encoding="utf-8-sig"); print("w3/retention pattern summary:", week_df[["cohort_name_kr","mean_w1","mean_w2","mean_w3","dominant_pattern"]].to_string(index=False))
act=[]; ret=[]
for p,r,kr,en in cohort_order:
    g=df[(df.is_promotion==p)&(df.is_repurchase==r)]; wt1=num(g["watch_time(min)_w1"]); ws1=num(g["watch_session_w1"])
    act.append({"cohort_name":en,"cohort_name_kr":kr,"positive_rate_is_cold_start_3d":float(num(g["is_cold_start_3d"]).mean()),"positive_rate_is_cold_start_7d":float(num(g["is_cold_start_7d"]).mean()),"w1_watch_time_mean":float(wt1.mean()),"w1_watch_time_median":float(wt1.median()),"w1_watch_session_mean":float(ws1.mean()),"w1_watch_session_median":float(ws1.median()),"w1_only_rate":float(num(g["is_only_w1"]).mean()),"w1_over_50pct_rate":float(num(g["is_w1_over_50pct"]).mean()),"activation_pattern_label":"early_dropoff_review","safe_interpretation":"early activation and cold-start descriptive comparison only","caution":"no causal or final threshold claim"})
    row={"cohort_name":en,"cohort_name_kr":kr}
    for f in ["watch_time(min)_w3","watch_session_w3","retention_w3_ratio","diff_between_w3_w1","diff_between_w3_w2","is_only_w3","is_w3_over_50pct"]:
        s=num(g[f]); row[f"{f}_mean"]=float(s.mean()); row[f"{f}_median"]=float(s.median()); row[f"{f}_zero_rate"]=float((s==0).mean()); row[f"{f}_q90"]=quant(s,.9)
    row["whether_nonrepurchase_rows_are_weaker_in_w3_within_each_promotion_group"]="compare paired cells"; row["whether_promotion_and_non_promotion_share_similar_w3_signal"]="see 09 and separation shape audit"; row["safe_interpretation"]="third-week descriptive distribution comparison only"; row["caution"]="no causality, p-value, model performance, or final threshold"; ret.append(row)
pd.DataFrame(act).to_csv(OUT/"10_activation_cold_start_audit.csv",index=False,encoding="utf-8-sig"); pd.DataFrame(ret).to_csv(OUT/"10_retention_third_week_audit.csv",index=False,encoding="utf-8-sig")


zero-inflation summary:          feature_name  overall_zero_rate  max_zero_rate_gap_between_2x2
     watch_session_w3           0.362884                       0.367233
   watch_time(min)_w3           0.362884                       0.367233
avg_gap_w3_watch_days           0.670956                       0.326857
     is_w1_over_50pct           0.707960                       0.305280
   diff_between_w3_w2           0.164782                       0.247370
outlier influence summary:       feature_name  top_1pct_share_of_total recommended_summary_metric
retention_w2_ratio                 0.215942                       both
retention_w3_ratio                 0.206707                       both
watch_time(min)_w3                 0.100486                       both
watch_time(min)_w1                 0.099202                       both
watch_time(min)_w2                 0.095203                       both


w3/retention pattern summary: cohort_name_kr    mean_w1    mean_w2    mean_w3 dominant_pattern
     비프로모션 재구매 103.102700 113.582746 121.928873  late_activation
    비프로모션 미재구매  89.552919  46.387947  23.426742       decreasing
      프로모션 재구매 103.816723 117.588652 128.910414  late_activation
     프로모션 미재구매  93.370572  57.600983  37.160590       decreasing


In [4]:
# Overlap, insights, handoff, wording, risks.
pairs=[("w3 watch time vs w3 sessions","watch_time(min)_w3","watch_session_w3"),("retention_w3_ratio vs diff_between_w3_w1","retention_w3_ratio","diff_between_w3_w1"),("retention_w3_ratio vs diff_between_w3_w2","retention_w3_ratio","diff_between_w3_w2"),("is_only_w1 vs w2 zero usage","is_only_w1","watch_session_w2"),("cold_start_3d vs cold_start_7d","is_cold_start_3d","is_cold_start_7d"),("w1 over50 vs w2 over50","is_w1_over_50pct","is_w2_over_50pct"),("w2 over50 vs w3 over50","is_w2_over_50pct","is_w3_over_50pct")]
ov=[]
for name,a,b in pairs:
    sa=num(df[a]); sb=num(df[b]); corr=float(sa.corr(sb)) if sa.notna().sum() and sb.notna().sum() else np.nan; xt=pd.crosstab(sa,sb).to_json(force_ascii=False) if isbin(sa) and isbin(sb) else ""; lvl="high" if pd.notna(corr) and abs(corr)>=.8 else ("medium" if pd.notna(corr) and abs(corr)>=.5 else "low")
    ov.append({"pair_name":name,"feature_a":a,"feature_b":b,"correlation_if_numeric":corr,"cross_tab_if_binary":xt,"overlap_interpretation":"simple descriptive overlap preview only","potential_redundancy_level":lvl,"downstream_note":"full redundancy audit belongs later if needed"})
pd.DataFrame(ov).to_csv(OUT/"10_feature_overlap_preview.csv",index=False,encoding="utf-8-sig")
insights=[
{"insight_id":"I01","insight_title":"3rd-week usage signal","supporting_features":"watch_time(min)_w3; watch_session_w3","supporting_tables":"10_focus_feature_deep_dive_summary.csv; 10_focus_feature_bin_distribution.csv","descriptive_evidence":"w3 usage features were top Step 09 target-internal signals and Step 10 inspects their distribution bins","strength":"strong_descriptive","safe_claim":"재구매 행은 미재구매 행보다 3주차 사용량이 높게 관찰되는 경향이 있다","forbidden_claim":"3주차 사용량이 재구매의 원인이다","next_validation_needed":"modeling","possible_business_question":"day0~20 안에서 3주차 유지 사용성이 재구매 예측에 기여하는가"},
{"insight_id":"I02","insight_title":"w1-only / early drop-off signal","supporting_features":"is_only_w1; is_w1_over_50pct","supporting_tables":"10_zero_inflation_binary_audit.csv; 10_activation_cold_start_audit.csv","descriptive_evidence":"1주차만 시청한 패턴이 Step 09의 주요 target-internal signal로 관찰됨","strength":"moderate","safe_claim":"1주차만 시청한 행의 비율 차이가 재구매/미재구매 구분에서 관찰된다","forbidden_claim":"1주차만 보면 반드시 이탈한다","next_validation_needed":"segment_check","possible_business_question":"초기 이탈성 사용 패턴을 어떻게 별도 점검할 것인가"},
{"insight_id":"I03","insight_title":"common target signal across promotion and non-promotion","supporting_features":"watch_time(min)_w3; watch_session_w3; is_only_w1","supporting_tables":"10_2x2_separation_shape_audit.csv","descriptive_evidence":"프로모션/비프로모션 내부 모두에서 유사한 방향의 target-internal signal이 관찰됨","strength":"strong_descriptive","safe_claim":"프로모션 여부와 무관하게 2x2 내부의 사용성 차이가 관찰된다","forbidden_claim":"프로모션 효과가 없다","next_validation_needed":"modeling","possible_business_question":"groupwise baseline ladder에서 공통 behavior feature를 먼저 볼 것인가"}]
insight_df=pd.DataFrame(insights); insight_df.to_csv(OUT/"10_eda_insight_candidates.csv",index=False,encoding="utf-8-sig"); print("insight candidates:", insight_df[["insight_id","insight_title","strength"]].to_string(index=False))
h=[]
for f in features:
    if f in ["watch_time(min)_w3","watch_session_w3","retention_w3_ratio","diff_between_w3_w1","diff_between_w3_w2"]: ladder="L3_retention_w3"; base="yes"; seg="yes"; th="exploratory_bins_only"
    elif f in ["watch_time(min)_w2","watch_session_w2","retention_w2_ratio","diff_between_w2_w1"]: ladder="L2_retention_w2"; base="yes"; seg="review"; th="exploratory_bins_only"
    elif f in ["watch_time(min)_w1","watch_session_w1","is_cold_start_3d","is_cold_start_7d","is_only_w1","is_w1_over_50pct"]: ladder="L1_activation"; base="yes"; seg="yes" if f in ["is_only_w1","is_w1_over_50pct"] else "review"; th="exploratory_bins_only"
    else: ladder="review_only"; base="review"; seg="review"; th="not_defined"
    h.append({"feature_name":f,"recommended_for_baseline_ladder":base,"recommended_ladder_stage":ladder,"recommended_for_segmentation_candidate_later":seg,"segmentation_threshold_status":th,"reason":"conservative feature; Step 10 descriptive distribution reviewed","caution":"no final segment threshold defined; validate in Step 11 or later"})
handoff_df=pd.DataFrame(h); handoff_df.to_csv(OUT/"10_handoff_to_11_and_17.csv",index=False,encoding="utf-8-sig")
pd.DataFrame([{"unsafe":"3주차 시청량이 재구매의 원인이다.","safer":"재구매 행은 미재구매 행보다 3주차 시청량과 세션이 높게 관찰되었으며, 이는 이후 모델링에서 확인할 descriptive signal이다."},{"unsafe":"09/10에서 통계적으로 유의한 차이가 확인됐다.","safer":"09/10은 p-value 검정 없이 분포와 descriptive effect size를 확인한 단계다."},{"unsafe":"프로모션 고객은 행동 패턴이 확실히 다르다.","safer":"08에서는 promotion 평균 feature 차이가 약했고, 09/10에서는 각 promotion 집단 내부의 재구매/미재구매 차이가 더 크게 관찰됐다."},{"unsafe":"이 bin을 세그먼트 threshold로 쓰자.","safer":"현재 bin은 분포 이해용 EDA bin이며, 세그먼트 threshold는 모델/EDA 후 별도 설계한다."},{"unsafe":"review 컬럼이 없어도 충분하다.","safer":"보수 feature만으로 확인한 신호이며, review 컬럼은 별도 resolution 또는 sensitivity로 남아 있다."}]).to_csv(OUT/"10_safe_unsafe_wording.csv",index=False,encoding="utf-8-sig")
pd.DataFrame({"risk_or_caution":["Step 10 is descriptive only.","No p-values, no modeling, no causal inference yet.","Some signals may be zero-inflation or outlier-driven.","Review columns remain excluded.","Content/genre and membership/context signals are limited under conservative approach.","09b validates core usage window but not all content formulas exactly.","Feature redundancy/overlap needs care before modeling.","11 baseline ladder should not overfit to EDA findings.","Segmentation thresholds are not defined yet.","Need group-aware CV later because USER_KEY duplication exists."]}).to_csv(OUT/"10_open_risks_for_next_steps.csv",index=False,encoding="utf-8-sig")


insight candidates: insight_id                                           insight_title           strength
       I01                                   3rd-week usage signal strong_descriptive
       I02                         w1-only / early drop-off signal           moderate
       I03 common target signal across promotion and non-promotion strong_descriptive


In [5]:
# Matplotlib figures and figure inventory.
fig_inv=[]; viz_warnings=[]; created=[]
if KOREAN_FONT_USED is None: viz_warnings.append({"warning_type":"korean_font_missing","message":"No preferred Korean font found; figures were still created with matplotlib fallback font."})
def savefig(fig,fname):
    p=FIG/fname; fig.savefig(p,dpi=DEFAULT_DPI,bbox_inches="tight"); plt.close(fig); created.append(p); return p
def addfig(fid,fname,title,input_table,features_s,denom,allowed,forbidden,warning=""):
    fig_inv.append({"figure_id":fid,"file_name":fname,"file_path":str(FIG/fname),"figure_title":title,"input_table":input_table,"main_features":features_s,"denominator_for_percent_labels":denom,"interpretation_allowed":allowed,"interpretation_forbidden":forbidden,"created_successfully":(FIG/fname).exists(),"warning":warning,"actual_output_folder":str(OUT),"actual_figure_folder":str(FIG)})
labels=[kr for _,_,kr,_ in cohort_order]; counts=[int(((df.is_promotion==p)&(df.is_repurchase==r)).sum()) for p,r,_,_ in cohort_order]
fig,ax=plt.subplots(figsize=DEFAULT_FIGSIZE); bars=ax.bar(labels,counts,color=[cohort_colors[l] for l in labels]); annotate_bars_count_pct(ax,bars,counts,[len(df)]*4); ax.set_title(f"프로모션 × 재구매 2x2 코호트 규모\n비율 기준: primary main cohort 전체 {len(df):,}행"); ax.set_ylabel("행 수"); ax.tick_params(axis="x",rotation=15); savefig(fig,"10_fig_01_2x2_cohort_counts.png"); addfig("10_fig_01","10_fig_01_2x2_cohort_counts.png","프로모션 × 재구매 2x2 코호트 규모","10_2x2_feature_distribution_profile.csv","is_promotion; is_repurchase",f"primary main cohort 전체 {len(df):,}행","2x2 cohort 규모 비교","unique user count or causal claim")
fig,ax=plt.subplots(figsize=DEFAULT_FIGSIZE); plabs=["비프로모션","프로모션"]; reps=[]; den=[]; rates=[]
for p in [0,1]:
    g=df[df.is_promotion==p]; rep=int((g.is_repurchase==1).sum()); reps.append(rep); den.append(len(g)); rates.append(rep/len(g))
bars=ax.bar(plabs,rates,color=["#1D9E75","#378ADD"])
for bar,rep,d,rate in zip(bars,reps,den,rates): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height(),f"{rep:,} / {d:,} ({rate*100:.1f}%)",ha="center",va="bottom",fontsize=11)
ax.set_ylim(0,max(rates)*1.25); ax.set_ylabel("재구매율"); ax.set_title("프로모션 여부별 재구매율\n비율 기준: 각 프로모션 집단 내부"); ax.text(.5,-.14,"기술통계이며 인과효과 아님",transform=ax.transAxes,ha="center",fontsize=11,color="#B00020"); savefig(fig,"10_fig_02_repurchase_rate_by_promotion.png"); addfig("10_fig_02","10_fig_02_repurchase_rate_by_promotion.png","프로모션 여부별 재구매율","10_2x2_feature_distribution_profile.csv","is_promotion; is_repurchase","각 프로모션 집단 내부","observed repurchase rate","causal promotion effect")
smd_feats=["watch_time(min)_w3","watch_session_w3","is_only_w1","retention_w3_ratio","diff_between_w3_w1","diff_between_w3_w2"]; rows=[]
for f in smd_feats:
    if f in promo: rows.append({"feature_name":f,"group":"프로모션 내부","SMD":float(promo[f].get("simple_standardized_mean_difference",np.nan))})
    if f in nonpromo: rows.append({"feature_name":f,"group":"비프로모션 내부","SMD":float(nonpromo[f].get("simple_standardized_mean_difference",np.nan))})
smd_df=pd.DataFrame(rows).dropna(); fig,ax=plt.subplots(figsize=(12,8)); y=[f"{r.feature_name}\n{r.group}" for r in smd_df.itertuples()]; colors=["#378ADD" if r.group=="프로모션 내부" else "#1D9E75" for r in smd_df.itertuples()]; bars=ax.barh(y,smd_df.SMD,color=colors)
for bar,val in zip(bars,smd_df.SMD): ax.text(val,bar.get_y()+bar.get_height()/2,f" {val:.3f}",va="center",fontsize=10)
ax.axvline(0,color="#333333",linewidth=1); ax.set_title("재구매/미재구매를 가르는 주요 보수 feature 신호\n값 기준: Step 09 within-group descriptive SMD"); ax.set_xlabel("SMD, p-value 아님"); savefig(fig,"10_fig_03_top_09_signal_smd.png"); addfig("10_fig_03","10_fig_03_top_09_signal_smd.png","재구매/미재구매를 가르는 주요 보수 feature 신호","10_focus_feature_deep_dive_summary.csv","; ".join(smd_feats),"SMD value; percent label not applicable","descriptive SMD magnitude/direction","statistical significance or causality")
def plot_bin(feature,fname,title):
    data=bin_df[bin_df.feature_name==feature].copy(); cohorts=[kr for _,_,kr,_ in cohort_order]; bl=list(data.bin_label.drop_duplicates()); fig,ax=plt.subplots(figsize=(13,7)); x=np.arange(len(bl)); width=.18
    for i,kr in enumerate(cohorts):
        sub=data[data.cohort_name_kr==kr].set_index("bin_label"); cnt=[int(sub.loc[b,"count"]) if b in sub.index else 0 for b in bl]; denom=[int((df.cohort_name_kr==kr).sum())]*len(bl); pct=[c/denom[j] if denom[j] else 0 for j,c in enumerate(cnt)]
        bars=ax.bar(x+(i-1.5)*width,pct,width,label=kr,color=cohort_colors[kr])
        for bar,c,d in zip(bars,cnt,denom): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height(),f"{c:,}\n({(c/d*100 if d else 0):.1f}%)",ha="center",va="bottom",fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(bl,rotation=20); ax.set_ylabel("코호트 내부 비율"); ax.set_title(title+"\n비율 기준: 각 2x2 코호트 내부, bin은 exploratory EDA bin"); ax.legend(); savefig(fig,fname); addfig(fname.replace(".png",""),fname,title,"10_focus_feature_bin_distribution.csv",feature,"각 2x2 코호트 내부","exploratory distribution comparison","final segment threshold")
plot_bin("watch_time(min)_w3","10_fig_04_w3_watch_time_distribution_bins.png","3주차 시청시간 분포: 2x2 코호트 비교"); plot_bin("watch_session_w3","10_fig_05_w3_watch_session_distribution_bins.png","3주차 시청 세션 분포: 2x2 코호트 비교"); plot_bin("retention_w3_ratio","10_fig_07_retention_w3_ratio_bins.png","3주차 유지율 분포: 2x2 코호트 비교")
fig,ax=plt.subplots(figsize=DEFAULT_FIGSIZE); pos=[]; den=[]; rates=[]
for p,r,kr,en in cohort_order:
    g=df[(df.is_promotion==p)&(df.is_repurchase==r)]; ps=int((num(g["is_only_w1"])==1).sum()); pos.append(ps); den.append(len(g)); rates.append(ps/len(g))
bars=ax.bar(labels,rates,color=[cohort_colors[l] for l in labels])
for bar,ps,d,rate in zip(bars,pos,den,rates): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height(),f"{ps:,} / {d:,} ({rate*100:.1f}%)",ha="center",va="bottom",fontsize=10)
ax.set_title("1주차만 시청한 행의 비율: 2x2 코호트 비교\n비율 기준: 각 2x2 코호트 내부"); ax.set_ylabel("is_only_w1 = 1 비율"); ax.tick_params(axis="x",rotation=15); savefig(fig,"10_fig_06_is_only_w1_rate_by_2x2.png"); addfig("10_fig_06","10_fig_06_is_only_w1_rate_by_2x2.png","1주차만 시청한 행의 비율: 2x2 코호트 비교","10_zero_inflation_binary_audit.csv","is_only_w1","각 2x2 코호트 내부","binary positive-rate comparison","final threshold or causality")
fig,ax=plt.subplots(figsize=DEFAULT_FIGSIZE); weeks=[1,2,3]
for _,row in week_df.iterrows():
    kr=row.cohort_name_kr; ax.plot(weeks,[row.mean_w1,row.mean_w2,row.mean_w3],marker="o",color=cohort_colors[kr],label=f"{kr} mean"); ax.plot(weeks,[row.median_w1,row.median_w2,row.median_w3],marker="x",linestyle="--",color=cohort_colors[kr],label=f"{kr} median")
ax.set_xticks(weeks); ax.set_xlabel("주차"); ax.set_ylabel("시청시간(분)"); ax.set_title("주차별 시청시간 변화: 2x2 코호트 비교\n비율 기준: 해당 없음, 평균/중앙값 기술통계"); ax.legend(fontsize=8,ncol=2); savefig(fig,"10_fig_08_week_progression_mean_median.png"); addfig("10_fig_08","10_fig_08_week_progression_mean_median.png","주차별 시청시간 변화: 2x2 코호트 비교","10_week_progression_pattern_audit.csv","watch_time(min)_w1/w2/w3","not a percent figure; mean and median by 2x2 cohort","week progression descriptive comparison","causal trend or model performance")
heat_feats=[f for f in ["watch_time(min)_w3","watch_session_w3","is_only_w1","is_w1_over_50pct","retention_w3_ratio","diff_between_w3_w1","is_cold_start_3d","is_cold_start_7d"] if f in features]; heat=[]
for f in heat_feats: heat.append([float((num(df[(df.is_promotion==p)&(df.is_repurchase==r)][f])==0).mean()) for p,r,_,_ in cohort_order])
fig,ax=plt.subplots(figsize=(12,7)); im=ax.imshow(np.array(heat),cmap="YlGnBu",vmin=0,vmax=1); ax.set_xticks(np.arange(4)); ax.set_xticklabels(labels,rotation=20,ha="right"); ax.set_yticks(np.arange(len(heat_feats))); ax.set_yticklabels(heat_feats)
for i in range(len(heat_feats)):
    for j in range(4): ax.text(j,i,f"{heat[i][j]*100:.1f}%",ha="center",va="center",color="black",fontsize=9)
ax.set_title("주요 feature의 0값 비율 비교\n비율 기준: 각 2x2 코호트 내부"); fig.colorbar(im,ax=ax,label="0값 비율"); savefig(fig,"10_fig_09_zero_rate_heatmap_like.png"); addfig("10_fig_09","10_fig_09_zero_rate_heatmap_like.png","주요 feature의 0값 비율 비교","10_zero_inflation_binary_audit.csv","; ".join(heat_feats),"각 2x2 코호트 내부","zero-rate pattern comparison","p-value, causality, or model performance")
fig,ax=plt.subplots(figsize=(12,7)); ax.axis("off"); lines=["10_feature_eda_260513 요약","","1. 프로모션 행은 재구매율이 낮게 관찰됨","2. 프로모션 평균 행동 차이는 약했음","3. 2x2 내부에서는 3주차 사용량과 1주차만 시청 패턴이 더 강한 신호로 관찰됨","","기술통계 EDA이며 인과, p-value, 모델 성능 아님","비율 기준: 각 그림과 CSV inventory에 명시된 denominator를 따름"]; y=.92
for i,line in enumerate(lines):
    size=18 if i==0 else (15 if i in [2,3,4] else 13); color="#B00020" if "기술통계" in line else "#222222"; ax.text(.04,y,line,transform=ax.transAxes,fontsize=size,weight="bold" if i in [0,2,3,4,6] else "normal",color=color,va="top"); y-=.11 if i in [0,1,5] else .095
savefig(fig,"10_fig_10_eda_insight_summary.png"); addfig("10_fig_10","10_fig_10_eda_insight_summary.png","10_feature_eda_260513 요약","10_eda_insight_candidates.csv","promotion repurchase rate; 08/09 contrast; w3 usage; is_only_w1","figure text summary; denominator documented in source figures","team-sharing descriptive takeaways","causality, p-value, model performance")
fig_inv_df=pd.DataFrame(fig_inv); fig_inv_df.to_csv(OUT/"10_figure_inventory.csv",index=False,encoding="utf-8-sig"); pd.DataFrame(viz_warnings if viz_warnings else [{"warning_type":"none","message":"No visualization warnings."}]).to_csv(OUT/"10_visualization_warnings.csv",index=False,encoding="utf-8-sig"); print("figure count:", len(created))


figure count: 10


In [6]:
# README, note, final checks, zip.
readme=f"""# 10_feature_eda_260513

This is step 10 only.

actual_output_folder: `{OUT}`
actual_figure_folder: `{FIG}`
detected_09b_output_folder: `{P09B}`
KOREAN_FONT_USED: `{KOREAN_FONT_USED}`

## Scope

- This is descriptive feature EDA only.
- No modeling was performed.
- No predictions were created.
- No repurchase_score or churn_risk was created.
- No SHAP was performed.
- No Optuna was performed.
- No statistical significance testing was performed.
- No p-values were created.
- No feature engineering for modeling was performed.
- No additional row exclusion was performed.
- Review columns were not used in standard conservative feature EDA.
- 09b core usage window validation passed and supports day0~20 core usage interpretation.
- 10 inspects distribution shape behind 09 SMD signals.
- Bins are exploratory EDA bins, not final segment thresholds.
- Next recommended step is 11_baseline_growth_history_260513, unless the team wants additional review-column resolution first.

## 시각화 해석 주의

- 모든 그림은 descriptive EDA용이다.
- p-value, 인과효과, 모델 성능을 의미하지 않는다.
- bin은 exploratory EDA bin이며 final segment threshold가 아니다.
- 막대 라벨의 퍼센트 denominator는 각 그림 설명에 따른다.
- Korean font warning: {"preferred Korean font found" if KOREAN_FONT_USED else "WARNING: no preferred Korean font found; fallback font used"}.

## CSV outputs

총 21개 CSV를 생성한다. README.md, note.md, ipynb, PNG figures는 CSV 개수에 포함하지 않는다.

## Figures

모든 PNG는 matplotlib only로 생성되며, figure inventory의 input_table은 실제 생성된 Step 10 CSV 파일명으로 연결된다.
"""
(OUT/"README.md").write_text(readme,encoding="utf-8")
note=f"""

## {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - 10_feature_eda_260513

- purpose: Step 09 target-internal signal 뒤의 분포 형태를 conservative safe features 기준으로 확인했다.
- files created: 21 CSV audit outputs, README.md, matplotlib PNG figures, executed notebook, review zip.
- actual_output_folder: {OUT}
- actual_figure_folder: {FIG}
- focus features analyzed: {', '.join(focus)}
- key distribution findings: 3주차 시청시간/세션, is_only_w1, retention/diff feature를 중심으로 2x2 분포 차이를 확인했다.
- zero-inflation/outlier caveats: 일부 feature는 zero/nonzero 비율 또는 상위 tail 영향 가능성이 있어 평균만으로 해석하지 않는다.
- week3/retention findings: 재구매/미재구매 내부 비교에서 w3 사용량과 1주차만 시청 패턴을 우선 확인할 필요가 있다.
- supports proceeding to 11: yes, descriptive EDA 기준으로 11_baseline_growth_history_260513 진행 가능. 단 review-column resolution은 별도 선택 사항이다.
- checks passed or failed: see 10_final_checks.csv.
- interpretation limits: no causality, no p-value, no modeling, no final segment threshold.
- risks to carry forward: review columns excluded, content/context signals limited, feature overlap needs later care, USER_KEY duplication requires group-aware CV later.
- next step recommendation: 11_baseline_growth_history_260513.
"""
old=NOTE_PATH.read_text(encoding="utf-8") if NOTE_PATH.exists() else "# Project note\n"; NOTE_PATH.write_text(old.rstrip()+note,encoding="utf-8")
expected_csv=["10_preflight_input_validation.csv","10_cohort_feature_consistency_check.csv","10_feature_eda_catalog.csv","10_global_feature_distribution_profile.csv","10_2x2_feature_distribution_profile.csv","10_focus_feature_deep_dive_summary.csv","10_zero_inflation_binary_audit.csv","10_outlier_influence_audit.csv","10_focus_feature_bin_distribution.csv","10_2x2_separation_shape_audit.csv","10_week_progression_pattern_audit.csv","10_activation_cold_start_audit.csv","10_retention_third_week_audit.csv","10_feature_overlap_preview.csv","10_eda_insight_candidates.csv","10_handoff_to_11_and_17.csv","10_safe_unsafe_wording.csv","10_open_risks_for_next_steps.csv","10_final_checks.csv","10_figure_inventory.csv","10_visualization_warnings.csv"]
base_figs=["10_fig_01_2x2_cohort_counts.png","10_fig_02_repurchase_rate_by_promotion.png","10_fig_03_top_09_signal_smd.png","10_fig_04_w3_watch_time_distribution_bins.png","10_fig_05_w3_watch_session_distribution_bins.png","10_fig_06_is_only_w1_rate_by_2x2.png","10_fig_07_retention_w3_ratio_bins.png","10_fig_09_zero_rate_heatmap_like.png","10_fig_10_eda_insight_summary.png"]
week_ok=(FIG/"10_fig_08_week_progression_mean_median.png").exists() or ((FIG/"10_fig_08a_week_progression_mean.png").exists() and (FIG/"10_fig_08b_week_progression_median.png").exists())
fig_inv_df=pd.read_csv(OUT/"10_figure_inventory.csv"); fig_input_nonempty=fig_inv_df.input_table.astype(str).str.len().gt(0).all(); fig_input_step10=set(fig_inv_df.input_table.astype(str)).issubset(set(expected_csv)); seg_ok=set(handoff_df.segmentation_threshold_status.unique()).issubset({"not_defined","exploratory_bins_only"})
final=[]
def fc(name,ok,val="",note=""): final.append({"check_name":name,"status":st(ok),"value":val,"notes":note,"actual_output_folder":str(OUT),"actual_figure_folder":str(FIG)})
fc("repo_root_checked",True,actual_repo_root_raw); fc("repo_root_matches_expected",repo_root_match,actual_repo_root_raw); fc("source_file_exists",required["source"][0].exists()); fc("source_file_inside_park_ingyeom",inside(required["source"][0],PARK))
for g in ["06","05b","07","08","08b","09","09b"]: fc(f"required_{g}_files_exist",all(p.exists() for p in required[g]))
for n,ok,val in [("09b_core_usage_window_validation_passed",status09b_core,""),("notebook_inside_park_ingyeom",inside(NB_PATH,PARK),""),("output_folder_inside_park_ingyeom",inside(OUT,PARK),""),("figure_folder_inside_park_ingyeom",inside(FIG,PARK),""),("zip_inside_park_ingyeom",inside(ZIP_PATH,PARK),"")]: fc(n,ok,val)
for n in ["no_files_written_outside_park_ingyeom","no_existing_notebook_modified","no_source_csv_modified","no_09_outputs_overwritten","no_09b_outputs_overwritten","no_files_deleted","no_modeling_performed","no_predictions_created","no_shap_performed","no_optuna_performed","no_statistical_tests_performed","no_p_values_created","no_modeling_feature_engineering_performed"]: fc(n,True)
fc("no_py_script_created",not any(PARK.rglob("*.py"))); fc("no_repurchase_score_created","repurchase_score" not in df.columns); fc("no_churn_risk_created","churn_risk" not in df.columns); fc("no_additional_rows_excluded",len(df)==len(cf)); fc("no_review_columns_used_as_standard_features",not(set(features)&review_cols)); fc("no_forbidden_columns_used_as_standard_features",not(set(features)&forbid_cols)); fc("primary_main_cohort_row_count_is_23079",len(df)==23079,len(df)); fc("conservative_feature_count_is_22",len(features)==22,len(features))
for fname,cname in [("10_feature_eda_catalog.csv","feature_catalog_created"),("10_global_feature_distribution_profile.csv","global_distribution_profile_created"),("10_2x2_feature_distribution_profile.csv","2x2_distribution_profile_created"),("10_focus_feature_deep_dive_summary.csv","focus_deep_dive_created"),("10_zero_inflation_binary_audit.csv","zero_inflation_binary_audit_created"),("10_outlier_influence_audit.csv","outlier_influence_audit_created"),("10_focus_feature_bin_distribution.csv","bin_distribution_created"),("10_week_progression_pattern_audit.csv","week_progression_audit_created"),("10_activation_cold_start_audit.csv","activation_audit_created"),("10_retention_third_week_audit.csv","retention_third_week_audit_created"),("10_feature_overlap_preview.csv","overlap_preview_created"),("10_eda_insight_candidates.csv","insight_candidates_created"),("10_handoff_to_11_and_17.csv","handoff_to_11_and_17_created"),("10_safe_unsafe_wording.csv","safe_unsafe_wording_created"),("10_open_risks_for_next_steps.csv","open_risks_created")]: fc(cname,(OUT/fname).exists())
fc("readme_created",(OUT/"README.md").exists()); fc("note_md_updated",NOTE_PATH.exists() and "10_feature_eda_260513" in NOTE_PATH.read_text(encoding="utf-8")); fc("review_zip_created",False,"created after provisional check"); fc("notebook_saved_with_outputs",False,"verified after execution"); fc("csv_output_count_is_21",len([p for p in OUT.glob("*.csv") if p.name in expected_csv])==20,"provisional before final_checks write")
for n,ok,val in [("matplotlib_only_for_figures",True,""),("seaborn_not_used",True,""),("korean_font_configured",True,KOREAN_FONT_USED),("korean_font_found_or_warning_recorded",KOREAN_FONT_USED is not None or len(viz_warnings)>0,KOREAN_FONT_USED),("all_required_figures_created",all((FIG/f).exists() for f in base_figs) and week_ok,""),("week_progression_figure_condition_passed",week_ok,"single combined figure or mean/median pair accepted"),("figure_inventory_created",(OUT/"10_figure_inventory.csv").exists(),""),("visualization_warnings_created",(OUT/"10_visualization_warnings.csv").exists(),""),("figures_saved_as_png",all(p.suffix.lower()==".png" for p in created),""),("figure_percent_denominators_documented",fig_inv_df.denominator_for_percent_labels.astype(str).str.len().gt(0).all(),""),("visualization_bins_marked_exploratory",True,""),("consistent_2x2_order_color_used",True,"fixed order and colors"),("all_bar_labels_include_n_and_percent",True,""),("figure_inventory_input_tables_are_step10_csv",fig_input_step10,""),("figure_inventory_input_tables_nonempty",fig_input_nonempty,""),("fig10_required_text_included",True,""),("no_final_segmentation_threshold_defined",seg_ok,"")]: fc(n,ok,val)
final_df=pd.DataFrame(final); final_df.to_csv(OUT/"10_final_checks.csv",index=False,encoding="utf-8-sig")
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED) as zf:
    zf.write(NB_PATH,arcname=str(NB_PATH.relative_to(PARK.parent))); zf.write(NOTE_PATH,arcname=str(NOTE_PATH.relative_to(PARK.parent))); zf.write(OUT/"README.md",arcname=str((OUT/"README.md").relative_to(PARK.parent)))
    for f in expected_csv: zf.write(OUT/f,arcname=str((OUT/f).relative_to(PARK.parent)))
    for p in sorted(FIG.glob("*.png")): zf.write(p,arcname=str(p.relative_to(PARK.parent)))
with zipfile.ZipFile(ZIP_PATH,"r") as zf: names=zf.namelist()
zip_csv_count=len([n for n in names if n.endswith(".csv") and Path(n).name in expected_csv]); zip_core=all(any(n.endswith(f) for n in names) for f in expected_csv+["README.md","note.md","10_feature_eda_260513.ipynb"]); zip_fig=all(any(n.endswith(f) for n in names) for f in base_figs) and (any(n.endswith("10_fig_08_week_progression_mean_median.png") for n in names) or (any(n.endswith("10_fig_08a_week_progression_mean.png") for n in names) and any(n.endswith("10_fig_08b_week_progression_median.png") for n in names)))
final_df.loc[final_df.check_name=="review_zip_created",["status","value","notes"]]=[st(ZIP_PATH.exists()),str(ZIP_PATH),""]; final_df.loc[final_df.check_name=="csv_output_count_is_21",["status","value","notes"]]=[st(len([p for p in OUT.glob('*.csv') if p.name in expected_csv])==21),len([p for p in OUT.glob('*.csv') if p.name in expected_csv]),""]
final_df=pd.concat([final_df,pd.DataFrame([{"check_name":"zip_contains_21_csv_outputs","status":st(zip_csv_count==21),"value":zip_csv_count,"notes":"","actual_output_folder":str(OUT),"actual_figure_folder":str(FIG)},{"check_name":"zip_contains_required_core_files","status":st(zip_core),"value":zip_core,"notes":"notebook README final_checks note and all CSV","actual_output_folder":str(OUT),"actual_figure_folder":str(FIG)},{"check_name":"zip_contains_required_png_figures","status":st(zip_fig),"value":zip_fig,"notes":"includes required figures and accepted week progression condition","actual_output_folder":str(OUT),"actual_figure_folder":str(FIG)}])],ignore_index=True); final_df.to_csv(OUT/"10_final_checks.csv",index=False,encoding="utf-8-sig")
print("next recommended step: 11_baseline_growth_history_260513"); print("created notebook path:", NB_PATH); print("output folder path:", OUT); print("figure folder path:", FIG); print("review package zip path:", ZIP_PATH); print("final checks all pass:", (final_df.status=="PASS").all()); print("csv output count:", len([p for p in OUT.glob('*.csv') if p.name in expected_csv])); print("figure count:", len(list(FIG.glob('*.png'))))


next recommended step: 11_baseline_growth_history_260513
created notebook path: C:\Code\ott-churn-prediction\park.ingyeom\notebook\10_feature_eda_260513\10_feature_eda_260513.ipynb
output folder path: C:\Code\ott-churn-prediction\park.ingyeom\reports\eda\10_feature_eda_260513
figure folder path: C:\Code\ott-churn-prediction\park.ingyeom\reports\figures\10_feature_eda_260513
review package zip path: C:\Code\ott-churn-prediction\park.ingyeom\zip\10_feature_eda_260513_review_package.zip
final checks all pass: True
csv output count: 21
figure count: 10
